In [1]:
# -*- coding: utf-8 -*-
"""
Sarcasm Detection with Ollama on Google Colab
"""

!pip install ollama pandas numpy

import pandas as pd
import time
import json
import re
import ollama
import subprocess
import requests
from threading import Thread
import os

MODEL_NAME = "deepseek-r1:1.5b"
INPUT_FILE = "/content/drive/MyDrive/Colab Notebooks/hotels.xlsx"
OUTPUT_FILE = "/content/drive/MyDrive/Colab Notebooks/hotels_sarcasm.xlsx"
DELAY_PER_REVIEW = 1.0

# Install and start Ollama
print("Installing Ollama...")
!curl -fsSL https://ollama.com/install.sh | sh

print("Starting Ollama server...")
# Start Ollama in the background
def start_ollama():
    subprocess.run(["ollama", "serve"])

# Start Ollama in a separate thread
ollama_thread = Thread(target=start_ollama)
ollama_thread.daemon = True
ollama_thread.start()

# Wait for Ollama to start
time.sleep(5)

print("Pulling the model...")
# Pull the model
!ollama pull {MODEL_NAME}

class SarcasmDetector:
    def __init__(self, model_name=MODEL_NAME):
        self.model = model_name
        self.prompt_template = """
        You are a sarcasm detection expert analyzing hotel reviews. Determine if this review contains sarcasm.
        Respond ONLY in JSON format with these exact keys:
        - "sarcasm": boolean (true/false)
        - "confidence": float between 0.0-1.0
        - "reason": brief explanation
        - "type": either "positive_sarcasm", "negative_sarcasm", or "none"

        Example Response:
        {{"sarcasm": true, "confidence": 0.85, "reason": "Uses positive words ironically", "type": "positive_sarcasm"}}

        Review: {review}
        """

    def analyze_review(self, review):
        try:
            prompt = self.prompt_template.format(review=review)

            # Retry mechanism for Ollama connection
            for attempt in range(3):
                try:
                    response = ollama.generate(
                        model=self.model,
                        prompt=prompt,
                        options={'temperature': 0.2, 'num_predict': 150}
                    )
                    json_str = response['response'].strip()

                    # Clean JSON response
                    if '```json' in json_str:
                        json_str = json_str.split('```json')[1].split('```')[0].strip()
                    elif '```' in json_str:
                        json_str = json_str.split('```')[1].strip()

                    json_match = re.search(r'\{[\s\S]*?\}', json_str)
                    if json_match:
                        json_str = json_match.group(0)
                        return json.loads(json_str)

                    break  # Success

                except Exception as e:
                    if attempt == 2:  # Last attempt
                        raise e
                    time.sleep(2)  # Wait before retry

            return self.heuristic_fallback(review)

        except Exception as e:
            print(f"Error processing review: {str(e)[:100]}")
            return self.heuristic_fallback(review)

    def heuristic_fallback(self, text):
        text_lower = text.lower()
        sarcastic_keywords = ['sarcastic', 'ironic', 'obviously', 'of course', 'surprise', 'shock', 'wow']
        positive_keywords = ['love', 'wonderful', 'perfect', 'amazing', 'excellent', 'fantastic', 'great']
        negative_keywords = ['terrible', 'awful', 'horrible', 'disgusting', 'worst']

        is_sarcastic = any(kw in text_lower for kw in sarcastic_keywords)
        confidence = 0.7 if is_sarcastic else 0.3

        if any(kw in text_lower for kw in positive_keywords):
            sarcasm_type = 'positive_sarcasm' if is_sarcastic else 'none'
        elif any(kw in text_lower for kw in negative_keywords):
            sarcasm_type = 'negative_sarcasm' if is_sarcastic else 'none'
        else:
            sarcasm_type = 'none'

        return {
            'sarcasm': is_sarcastic,
            'confidence': confidence,
            'reason': f"Heuristic fallback: {text[:100]}...",
            'type': sarcasm_type
        }

    def process_dataframe(self, df, review_column='Review Summary'):
        print(f"Processing {len(df)} reviews...")

        # Initialize new columns
        df['sarcasm'] = False
        df['sarcasm_confidence'] = 0.0
        df['sarcasm_reason'] = ''
        df['sarcasm_type'] = 'none'

        for idx, row in df.iterrows():
            review_text = row[review_column]
            if not isinstance(review_text, str) or len(review_text.strip()) == 0:
                continue

            try:
                if idx % 5 == 0:
                    print(f"Processing review {idx + 1}/{len(df)}")

                result = self.analyze_review(review_text)

                df.at[idx, 'sarcasm'] = result.get('sarcasm', False)
                df.at[idx, 'sarcasm_confidence'] = result.get('confidence', 0.0)
                df.at[idx, 'sarcasm_reason'] = str(result.get('reason', ''))[:200]
                df.at[idx, 'sarcasm_type'] = result.get('type', 'none')

                # Save progress every 20 reviews
                if idx % 20 == 0:
                    df.to_excel(OUTPUT_FILE, index=False)
                    print(f"Progress saved at review {idx + 1}")

            except Exception as e:
                print(f"Error on row {idx}: {str(e)[:100]}")

            time.sleep(DELAY_PER_REVIEW)

        return df


def main():
    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive')

    # Check if Ollama is running
    try:
        result = subprocess.run(["ollama", "list"], capture_output=True, text=True)
        if result.returncode == 0:
            print("Ollama is running successfully")
            print("Available models:")
            print(result.stdout)
        else:
            print("Ollama not running properly, trying to restart...")
            subprocess.run(["pkill", "ollama"])
            time.sleep(2)
            subprocess.run(["ollama", "serve"], check=True)
            time.sleep(5)
    except Exception as e:
        print(f"Ollama check failed: {str(e)}")
        return

    try:
        df = pd.read_excel(INPUT_FILE)
        print(f"Loaded {len(df)} reviews from {INPUT_FILE}")

        # For testing, use a smaller subset
        if len(df) > 100:
            print("Using first 100 reviews for testing...")
            df = df.head(100).copy()

    except Exception as e:
        print(f"Error loading file: {str(e)}")
        print("Please check if the file exists at:", INPUT_FILE)
        return

    print("Initializing sarcasm detector...")
    detector = SarcasmDetector()

    print("Detecting sarcasm in reviews...")
    start_time = time.time()
    df = detector.process_dataframe(df)
    proc_time = time.time() - start_time

    print(f"Completed in {proc_time:.2f} seconds")
    print(f"Processing rate: {len(df) / proc_time:.2f} reviews/second")

    sarcastic_count = df[df['sarcasm']].shape[0]
    high_conf_count = df[df['sarcasm_confidence'] > 0.7].shape[0]

    print(f"\nResults")
    print("=" * 50)
    print(f"Total reviews: {len(df)}")
    print(f"Sarcastic reviews detected: {sarcastic_count} ({sarcastic_count / len(df) * 100:.1f}%)")
    print(f"High confidence detections (>0.7): {high_conf_count}")

    print("\nSample sarcastic reviews:")
    print("=" * 50)
    sarcastic_samples = df[df['sarcasm']].head(3)
    for idx, row in sarcastic_samples.iterrows():
        print(f"\nReview: {row['Review Summary'][:200]}...")
        print(f"Confidence: {row['sarcasm_confidence']:.2f}")
        print(f"Type: {row['sarcasm_type']}")
        print(f"Reason: {row['sarcasm_reason']}")
        print("-" * 50)

    # Save final results
    df.to_excel(OUTPUT_FILE, index=False)
    print(f"Saved to {OUTPUT_FILE}")


if __name__ == "__main__":
    main()

Installing Ollama...
>>> Installing ollama to /usr/local
>>> Downloading Linux amd64 bundle
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
Starting Ollama server...
Pulling the model...

Mounted at /content/drive
Ollama is running successfully
Available models:
NAME                ID              SIZE      MODIFIED       
deepseek-r1:1.5b    e0979632db5a    1.1 GB    29 seconds ago    

Loaded 6780 reviews from /content/drive/MyDrive/Colab Notebooks/hotels.xlsx
Using first 100 reviews for testing...
Initializing sarcasm detector...
Detecting sarcasm in reviews...
Processing 100 reviews...
Processing review 1/100
Progress saved at review 1
Processing review 6/100
Processing review 11/100
Proces

In [2]:
# -*- coding: utf-8 -*-
"""
Sarcasm Detection with Ollama on Google Colab
"""

!pip install ollama pandas numpy
!pip install emot

import pandas as pd
import time
import json
import re
import ollama
import subprocess
import numpy as np
from threading import Thread
from emot.emo_unicode import EMOTICONS_EMO
import warnings
warnings.filterwarnings('ignore')

MODEL_NAME = "deepseek-r1:1.5b"
INPUT_FILE = "/content/drive/MyDrive/Colab Notebooks/hotels.xlsx"
OUTPUT_FILE = "/content/drive/MyDrive/Colab Notebooks/hotels_sarcasm.xlsx"
DELAY_PER_REVIEW = 0.1  # Reduced delay for faster processing

# Install and start Ollama
print("Installing Ollama...")
!curl -fsSL https://ollama.com/install.sh | sh

print("Starting Ollama server...")
# Start Ollama in the background
def start_ollama():
    subprocess.run(["ollama", "serve"])

# Start Ollama in a separate thread
ollama_thread = Thread(target=start_ollama)
ollama_thread.daemon = True
ollama_thread.start()

# Wait for Ollama to start
time.sleep(5)

print("Pulling the model...")
# Pull the model
!ollama pull {MODEL_NAME}

# =============================================================================
# PREPROCESSING FUNCTIONS FOR SARCASM DETECTION
# =============================================================================
def handle_emoticons(text):
    """Efficient emoticon handling with precompiled patterns"""
    if not hasattr(handle_emoticons, 'emoticon_patterns'):
        # Precompile emoticon patterns for efficiency
        handle_emoticons.emoticon_patterns = []
        for emoticon, description in EMOTICONS_EMO.items():
            # Escape special regex characters in emoticons
            escaped_emoticon = re.escape(emoticon)
            handle_emoticons.emoticon_patterns.append(
                (re.compile(escaped_emoticon), f" {description} ")
            )

    text = str(text)
    for pattern, replacement in handle_emoticons.emoticon_patterns:
        text = pattern.sub(replacement, text)
    return text

def preprocess_for_sarcasm(text):
    """Specialized preprocessing for sarcasm detection - preserves linguistic cues"""
    if pd.isna(text) or text == "":
        return ""

    text = str(text)

    # Basic cleaning only - preserve case and punctuation!
    text = re.sub(r'http\S+|www\.\S+', '', text)  # Remove URLs
    text = re.sub(r'\S+@\S+', '', text)  # Remove emails
    text = re.sub(r'\s+', ' ', text)  # Remove extra spaces

    # Handle emoticons (CRUCIAL for sarcasm detection)
    text = handle_emoticons(text)

    return text.strip()

# =============================================================================
# SARCASM DETECTOR CLASS
# =============================================================================
class SarcasmDetector:
    def __init__(self, model_name=MODEL_NAME):
        self.model = model_name
        self.prompt_template = """
        You are a sarcasm detection expert analyzing hotel reviews. Determine if this review contains sarcasm.
        Respond ONLY in JSON format with these exact keys:
        - "sarcasm": boolean (true/false)
        - "confidence": float between 0.0-1.0
        - "reason": brief explanation
        - "type": either "positive_sarcasm", "negative_sarcasm", or "none"

        Example Response:
        {{"sarcasm": true, "confidence": 0.85, "reason": "Uses positive words ironically", "type": "positive_sarcasm"}}

        Review: {review}
        """

    def analyze_review(self, review):
        try:
            prompt = self.prompt_template.format(review=review)

            # Retry mechanism for Ollama connection
            for attempt in range(3):
                try:
                    response = ollama.generate(
                        model=self.model,
                        prompt=prompt,
                        options={'temperature': 0.2, 'num_predict': 150}
                    )
                    json_str = response['response'].strip()

                    # Clean JSON response
                    if '```json' in json_str:
                        json_str = json_str.split('```json')[1].split('```')[0].strip()
                    elif '```' in json_str:
                        json_str = json_str.split('```')[1].strip()

                    json_match = re.search(r'\{[\s\S]*?\}', json_str)
                    if json_match:
                        json_str = json_match.group(0)
                        return json.loads(json_str)

                    break  # Success

                except Exception as e:
                    if attempt == 2:  # Last attempt
                        raise e
                    time.sleep(2)  # Wait before retry

            return self.heuristic_fallback(review)

        except Exception as e:
            print(f"Error processing review: {str(e)[:100]}")
            return self.heuristic_fallback(review)

    def heuristic_fallback(self, text):
        text_lower = text.lower()
        sarcastic_keywords = [
            'sarcastic', 'ironic', 'irony', 'obviously', 'of course',
            'surprise', 'shock', 'wow', 'really', 'just what i wanted',
            'thanks for', 'great job', 'lovely', 'fantastic', 'perfect',
            'amazing', 'wonderful', 'excellent', 'brilliant', 'outstanding'
        ]

        positive_keywords = ['love', 'wonderful', 'perfect', 'amazing', 'excellent', 'fantastic', 'great', 'best']
        negative_keywords = ['terrible', 'awful', 'horrible', 'disgusting', 'worst', 'never again', 'avoid']

        # Check for exaggerated language patterns
        has_exaggeration = any(word in text_lower for word in sarcastic_keywords)
        has_multiple_exclamations = text.count('!') >= 3
        has_ellipsis = '...' in text
        has_sarcastic_quotes = '\"' in text and any(word in text_lower for word in ['so', 'very', 'really'])

        is_sarcastic = has_exaggeration or has_multiple_exclamations or has_ellipsis or has_sarcastic_quotes
        confidence = 0.7 if is_sarcastic else 0.3

        # Determine sarcasm type
        if any(kw in text_lower for kw in positive_keywords):
            sarcasm_type = 'positive_sarcasm' if is_sarcastic else 'none'
        elif any(kw in text_lower for kw in negative_keywords):
            sarcasm_type = 'negative_sarcasm' if is_sarcastic else 'none'
        else:
            sarcasm_type = 'none'

        return {
            'sarcasm': is_sarcastic,
            'confidence': confidence,
            'reason': f"Heuristic fallback: detected patterns indicative of sarcasm",
            'type': sarcasm_type
        }

    def process_dataframe(self, df, review_column='Review Summary'):
        print(f"Processing {len(df)} reviews...")

        # Store original columns
        original_columns = df.columns.tolist()

        # Initialize new columns - use 0/1 for sarcasm instead of True/False
        df['sarcasm'] = 0  # 0 for not sarcastic, 1 for sarcastic
        df['sarcasm_confidence'] = 0.0
        df['sarcasm_reason'] = ''
        df['sarcasm_type'] = 'none'
        df['processed_review'] = ''

        for idx, row in df.iterrows():
            review_text = row[review_column]
            if not isinstance(review_text, str) or len(review_text.strip()) == 0:
                continue

            try:
                if idx % 5 == 0:
                    print(f"Processing review {idx + 1}/{len(df)}")

                # Preprocess for sarcasm detection (light cleaning)
                processed_text = preprocess_for_sarcasm(review_text)
                df.at[idx, 'processed_review'] = processed_text

                result = self.analyze_review(processed_text)

                # Convert boolean to 0/1
                sarcasm_bool = result.get('sarcasm', False)
                df.at[idx, 'sarcasm'] = 1 if sarcasm_bool else 0
                df.at[idx, 'sarcasm_confidence'] = result.get('confidence', 0.0)
                df.at[idx, 'sarcasm_reason'] = str(result.get('reason', ''))[:200]
                df.at[idx, 'sarcasm_type'] = result.get('type', 'none')

                # Save progress every 20 reviews
                if idx % 20 == 0:
                    # Save with all original columns + new sarcasm columns
                    df.to_excel(OUTPUT_FILE, index=False)
                    print(f"Progress saved at review {idx + 1}")

            except Exception as e:
                print(f"Error on row {idx}: {str(e)[:100]}")

            time.sleep(DELAY_PER_REVIEW)

        return df

# =============================================================================
# MAIN EXECUTION
# =============================================================================
def main():
    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive')

    # Check if Ollama is running
    try:
        result = subprocess.run(["ollama", "list"], capture_output=True, text=True, timeout=10)
        if result.returncode == 0:
            print("Ollama is running successfully")
        else:
            print("Ollama not running properly, trying to continue...")
    except Exception as e:
        print(f"Ollama check: {str(e)}")
        print("Trying to continue...")

    try:
        df = pd.read_excel(INPUT_FILE)
        print(f"Loaded {len(df)} reviews from {INPUT_FILE}")
        print(f"Original columns: {df.columns.tolist()}")

        # For testing, use a smaller subset (remove this for full processing)
        if len(df) > 30:
            print("Using first 30 reviews for testing...")
            df = df.head(30).copy()

    except Exception as e:
        print(f"Error loading file: {str(e)}")
        print("Please check if the file exists at:", INPUT_FILE)
        return

    print("Initializing sarcasm detector...")
    detector = SarcasmDetector()

    print("Detecting sarcasm in reviews...")
    start_time = time.time()
    df = detector.process_dataframe(df)
    proc_time = time.time() - start_time

    print(f"Completed in {proc_time:.2f} seconds")
    print(f"Processing rate: {len(df) / proc_time:.2f} reviews/second")

    sarcastic_count = df[df['sarcasm'] == 1].shape[0]
    high_conf_count = df[df['sarcasm_confidence'] > 0.7].shape[0]

    print(f"\nResults")
    print("=" * 50)
    print(f"Total reviews: {len(df)}")
    print(f"Sarcastic reviews detected (sarcasm=1): {sarcastic_count} ({sarcastic_count / len(df) * 100:.1f}%)")
    print(f"Non-sarcastic reviews (sarcasm=0): {len(df) - sarcastic_count} ({(len(df) - sarcastic_count) / len(df) * 100:.1f}%)")
    print(f"High confidence detections (>0.7): {high_conf_count}")

    print("\nSarcasm type distribution:")
    print("=" * 50)
    for sarcasm_type in df['sarcasm_type'].unique():
        count = (df['sarcasm_type'] == sarcasm_type).sum()
        print(f"{sarcasm_type}: {count} ({count/len(df)*100:.1f}%)")

    # Save final results with ALL original columns + new sarcasm columns
    df.to_excel(OUTPUT_FILE, index=False)
    print(f"Saved to {OUTPUT_FILE}")

    # Show final output structure
    print(f"\nOutput file contains {len(df.columns)} columns:")
    for i, col in enumerate(df.columns, 1):
        print(f"{i}. {col}")

    print(f"\nSample output:")
    sample_cols = ['Hotel name', 'Review Summary', 'sarcasm', 'sarcasm_confidence', 'sarcasm_type']
    available_cols = [col for col in sample_cols if col in df.columns]
    print(df[available_cols].head(3))


if __name__ == "__main__":
    main()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.5/61.5 kB 4.7 MB/s eta 0:00:00
Installing Ollama...
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading Linux amd64 bundle
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
Starting Ollama server...
Pulling the model...

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Ollama is running successfully
Loaded 6780 reviews from /content/drive/MyDrive/Colab Notebooks/hotels.xlsx
Original columns: ['Hotel name', ' Guest Name ', ' Traveller Type ', ' Review Date ', ' Rating /10 ', ' Liked Categories ', 'Review Summary', ' Stay Duration ', ' Stay Month ',